In [1]:
import os
import pickle
import sqlite3
import time

import numpy as np
import pandas as pd
import joblib
import faiss
import shap
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(
    "processed/telco_eda_processed.csv"
)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,MonthlyChargeGroup,NumberOfServices,HighValueCustomer,CustomerSegment
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Yes,Electronic check,29.85,29.85,No,0-6 Months,Low,1,False,New Customer
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,Mailed check,56.95,1889.50,No,25-48 Months,Medium,3,False,Regular Customer
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,Mailed check,53.85,108.15,Yes,0-6 Months,Medium,3,False,New Customer
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,Bank transfer (automatic),42.30,1840.75,No,25-48 Months,Medium,3,False,Regular Customer
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,Electronic check,70.70,151.65,Yes,0-6 Months,High,1,False,New Customer


In [3]:
df = pd.read_csv(
    "processed/telco_eda_processed.csv"
)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,MonthlyChargeGroup,NumberOfServices,HighValueCustomer,CustomerSegment
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Yes,Electronic check,29.85,29.85,No,0-6 Months,Low,1,False,New Customer
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,Mailed check,56.95,1889.50,No,25-48 Months,Medium,3,False,Regular Customer
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,Mailed check,53.85,108.15,Yes,0-6 Months,Medium,3,False,New Customer
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,Bank transfer (automatic),42.30,1840.75,No,25-48 Months,Medium,3,False,Regular Customer
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,Electronic check,70.70,151.65,Yes,0-6 Months,High,1,False,New Customer


In [4]:
customer_df = df.copy()

customer_df["NewCustomer"] = (
    customer_df["tenure"] <= 6
).astype(int)

customer_df["LongTermCustomer"] = (
    customer_df["tenure"] >= 48
).astype(int)

customer_df["ChargeToTenure"] = (
    customer_df["TotalCharges"] /
    (customer_df["tenure"] + 1)
)

monthly_threshold = customer_df[
    "MonthlyCharges"
].quantile(0.75)

customer_df["HighMonthlyCharge"] = (
    customer_df["MonthlyCharges"] >= monthly_threshold
).astype(int)

customer_df["HighValueHighRisk"] = (
    (
        customer_df["MonthlyCharges"] >= monthly_threshold
    )
    &
    (
        customer_df["tenure"] <= 6
    )
).astype(int)

customer_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TenureGroup,MonthlyChargeGroup,NumberOfServices,HighValueCustomer,CustomerSegment,NewCustomer,LongTermCustomer,ChargeToTenure,HighMonthlyCharge,HighValueHighRisk
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,0-6 Months,Low,1,False,New Customer,1,0,14.925000,0,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,25-48 Months,Medium,3,False,Regular Customer,0,0,53.985714,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,0-6 Months,Medium,3,False,New Customer,1,0,36.050000,0,0
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,25-48 Months,Medium,3,False,Regular Customer,0,0,40.016304,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,0-6 Months,High,1,False,New Customer,1,0,50.550000,0,0


In [5]:
preprocessor = joblib.load(
    "../models/preprocessor.pkl"
)

print("Preprocessor loaded.")

Preprocessor loaded.


In [6]:
model_path = "../models/improved_logistic_regression.pkl"

model = joblib.load(model_path)

print("ML model loaded.")
print(model)

ML model loaded.
LogisticRegression(C=0.003593813663804626, class_weight={0: 1, 1: 2.5},
                   max_iter=2000, random_state=42, solver='liblinear')


In [7]:
target = "Churn"

feature_df = customer_df.drop(
    columns=["Churn", "customerID"],
    errors="ignore"
)

X_all = feature_df.copy()

print("Feature shape:", X_all.shape)

Feature shape: (7043, 29)


In [8]:
X_all_processed = preprocessor.transform(
    X_all
)

print(
    "Processed feature shape:",
    X_all_processed.shape
)

Processed feature shape: (7043, 65)


In [9]:
CHURN_THRESHOLD = 0.55

def predict_customer(customer_id):
    
    customer = customer_df[
        customer_df["customerID"] == customer_id
    ]
    
    if customer.empty:
        return {
            "error": "Customer ID not found."
        }
    
    X_customer = customer.drop(
        columns=["Churn", "customerID"],
        errors="ignore"
    )
    
    X_customer_processed = preprocessor.transform(
        X_customer
    )
    
    probability = model.predict_proba(
        X_customer_processed
    )[0, 1]
    
    prediction = int(
        probability >= CHURN_THRESHOLD
    )
    
    risk = (
        "High"
        if probability >= CHURN_THRESHOLD
        else "Low"
    )
    
    return {
        "customer_id": customer_id,
        "churn_probability": float(probability),
        "churn_prediction": prediction,
        "risk_level": risk
    }

In [10]:
test_customer_id = customer_df[
    "customerID"
].iloc[0]

prediction = predict_customer(
    test_customer_id
)

prediction

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


{'customer_id': '7590-VHVEG',
 'churn_probability': 0.7567963733310893,
 'churn_prediction': 1,
 'risk_level': 'High'}

In [11]:
def get_customer_profile(customer_id):
    
    customer = customer_df[
        customer_df["customerID"] == customer_id
    ]
    
    if customer.empty:
        return None
    
    row = customer.iloc[0]
    
    profile = {
        "customerID": row["customerID"],
        "gender": row["gender"],
        "SeniorCitizen": row["SeniorCitizen"],
        "Partner": row["Partner"],
        "Dependents": row["Dependents"],
        "tenure": row["tenure"],
        "PhoneService": row["PhoneService"],
        "InternetService": row["InternetService"],
        "Contract": row["Contract"],
        "PaymentMethod": row["PaymentMethod"],
        "MonthlyCharges": row["MonthlyCharges"],
        "TotalCharges": row["TotalCharges"],
        "PaperlessBilling": row["PaperlessBilling"]
    }
    
    return profile

In [12]:
profile = get_customer_profile(
    test_customer_id
)

profile

{'customerID': '7590-VHVEG',
 'gender': 'Female',
 'SeniorCitizen': 0,
 'Partner': 'Yes',
 'Dependents': 'No',
 'tenure': 1,
 'PhoneService': 'No',
 'InternetService': 'DSL',
 'Contract': 'Month-to-month',
 'PaymentMethod': 'Electronic check',
 'MonthlyCharges': 29.85,
 'TotalCharges': 29.85,
 'PaperlessBilling': 'Yes'}

In [13]:
shap_explainer = shap.TreeExplainer(
    joblib.load(
        "../models/xgboost_churn_model.pkl"
    )
)

print("SHAP explainer loaded.")

SHAP explainer loaded.


In [14]:
logistic_model = model

background_size = min(
    500,
    len(X_all_processed)
)

background_data = X_all_processed[
    :background_size
]

linear_explainer = shap.LinearExplainer(
    logistic_model,
    background_data
)

print("Logistic Regression SHAP explainer created.")

Logistic Regression SHAP explainer created.


In [15]:
def get_shap_explanation(
    customer_id,
    top_n=5
):
    
    customer = customer_df[
        customer_df["customerID"] == customer_id
    ]
    
    if customer.empty:
        return None
    
    X_customer = customer.drop(
        columns=["Churn", "customerID"],
        errors="ignore"
    )
    
    X_processed = preprocessor.transform(
        X_customer
    )
    
    shap_values = linear_explainer(
        X_processed
    )
    
    values = shap_values.values[0]
    
    feature_names = preprocessor.get_feature_names_out()
    
    shap_table = pd.DataFrame({
        "Feature": feature_names,
        "SHAP_Value": values,
        "Absolute_SHAP": np.abs(values)
    })
    
    shap_table = shap_table.sort_values(
        "Absolute_SHAP",
        ascending=False
    )
    
    return shap_table.head(top_n)

In [16]:
shap_result = get_shap_explanation(
    test_customer_id,
    top_n=5
)

shap_result

,Feature,SHAP_Value,Absolute_SHAP
5,num__NewCustomer,0.436900,0.436900
1,num__tenure,0.400227,0.400227
2,num__MonthlyCharges,-0.312355,0.312355
21,cat__InternetService_DSL,-0.153317,0.153317
42,cat__Contract_Month-to-month,0.142881,0.142881


In [17]:
def get_shap_reasons(
    customer_id,
    top_n=5
):
    
    shap_table = get_shap_explanation(
        customer_id,
        top_n
    )
    
    if shap_table is None:
        return []
    
    positive = shap_table[
        shap_table["SHAP_Value"] > 0
    ]
    
    negative = shap_table[
        shap_table["SHAP_Value"] < 0
    ]
    
    reasons = {
        "positive_contributors":
            positive[
                ["Feature", "SHAP_Value"]
            ].to_dict("records"),
        
        "negative_contributors":
            negative[
                ["Feature", "SHAP_Value"]
            ].to_dict("records")
    }
    
    return reasons

In [18]:
shap_reasons = get_shap_reasons(
    test_customer_id
)

shap_reasons

{'positive_contributors': [{'Feature': 'num__NewCustomer',
   'SHAP_Value': 0.4369004741034778},
  {'Feature': 'num__tenure', 'SHAP_Value': 0.4002273412449052},
  {'Feature': 'cat__Contract_Month-to-month',
   'SHAP_Value': 0.14288083157133452}],
 'negative_contributors': [{'Feature': 'num__MonthlyCharges',
   'SHAP_Value': -0.31235461527130465},
  {'Feature': 'cat__InternetService_DSL', 'SHAP_Value': -0.1533174411621327}]}

In [19]:
knowledge_index = faiss.read_index(
    "../vector_store/knowledge_faiss.index"
)

with open(
    "../vector_store/knowledge_metadata.pkl",
    "rb"
) as f:
    knowledge_df = pickle.load(f)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Knowledge vectors:",
    knowledge_index.ntotal
)

Knowledge vectors: 6


In [20]:
def search_knowledge_base(
    query,
    top_k=3
):
    
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )
    
    scores, indices = knowledge_index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )
    
    results = []
    
    for score, idx in zip(
        scores[0],
        indices[0]
    ):
        
        if idx < len(knowledge_df):
            
            row = knowledge_df.iloc[idx]
            
            results.append({
                "source": row["source"],
                "text": row["text"],
                "score": float(score)
            })
    
    return results

In [30]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

llm.eval()

print("Local LLM loaded.")

Local LLM loaded.


In [31]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

llm.eval()

print("Local LLM loaded.")

Local LLM loaded.


In [32]:
def build_customer_context(customer_id):
    
    profile = get_customer_profile(
        customer_id
    )
    
    prediction = predict_customer(
        customer_id
    )
    
    shap_reasons = get_shap_reasons(
        customer_id
    )
    
    if profile is None:
        return None
    
    context = f"""
CUSTOMER PROFILE:
{profile}

ML CHURN RESULT:
{prediction}

SHAP EXPLANATION:
{shap_reasons}
"""
    
    return context

In [33]:
def build_integrated_context(
    customer_id,
    question
):
    
    customer_context = build_customer_context(
        customer_id
    )
    
    knowledge_results = search_knowledge_base(
        question,
        top_k=3
    )
    
    knowledge_context = "\n\n".join(
        [
            f"Source: {r['source']}\n{r['text']}"
            for r in knowledge_results
        ]
    )
    
    return f"""
VERIFIED CUSTOMER INFORMATION:

{customer_context}

VERIFIED BUSINESS KNOWLEDGE:

{knowledge_context}
"""

In [34]:
def build_integrated_prompt(
    customer_id,
    question
):
    
    context = build_integrated_context(
        customer_id,
        question
    )
    
    prompt = f"""
You are an AI Customer Intelligence Assistant.

Answer the user's question using ONLY the verified information provided below.

STRICT RULES:
- Never invent customer information.
- Never invent churn probabilities.
- Never change the ML prediction.
- Never invent SHAP explanations.
- Never invent company policies.
- Do not perform your own numerical prediction.
- If information is unavailable, say so.
- Clearly distinguish customer facts from business policies.
- Keep the response concise and useful.

VERIFIED SYSTEM INFORMATION:
{context}

CUSTOMER ID:
{customer_id}

USER QUESTION:
{question}

FINAL ANSWER:
"""
    
    return prompt

In [35]:
def generate_integrated_answer(
    customer_id,
    question,
    max_new_tokens=180
):
    
    prompt = build_integrated_prompt(
        customer_id,
        question
    )
    
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )
    
    start_time = time.time()
    
    with torch.no_grad():
        
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    elapsed = time.time() - start_time
    
    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]
    
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )
    
    return {
        "customer_id": customer_id,
        "question": question,
        "answer": answer.strip(),
        "response_time": elapsed
    }

In [37]:
question = """
Why is this customer considered at risk,
and what retention policy should support staff consider?
"""

result = generate_integrated_answer(
    test_customer_id,
    question
)

print("CUSTOMER:", result["customer_id"])
print("\nQUESTION:")
print(result["question"])

print("\nAI ANSWER:")
print(result["answer"])

print(
    "\nRESPONSE TIME:",
    round(result["response_time"], 2),
    "seconds"
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CUSTOMER: 7590-VHVEG

QUESTION:

Why is this customer considered at risk,
and what retention policy should support staff consider?


AI ANSWER:
This customer is considered at high risk of churning due to several factors, including their tenure (1 month), monthly charges ($29.85), and payment method (Electronic Check). The churn probability of 0.7568 indicates a significant likelihood of leaving within the next year.

Retention policy considerations for this customer might include:

1. **Reviewing Services**: Ensure there are no issues with the services being offered.
2. **Understanding Billing Concerns**: Address any billing questions or discrepancies promptly.
3. **Contract Conditions**: Verify that the customer understands all terms and conditions of the contract.
4. **Technical Problems**: Investigate and resolve any reported technical issues.
5. **Support**: Offer immediate assistance and ensure the customer feels supported.

The AI system has flagged potential risk factors such as

In [38]:
question = """
Why is this customer considered at risk,
and what retention policy should support staff consider?
"""

result = generate_integrated_answer(
    test_customer_id,
    question
)

print("CUSTOMER:", result["customer_id"])
print("\nQUESTION:")
print(result["question"])

print("\nAI ANSWER:")
print(result["answer"])

print(
    "\nRESPONSE TIME:",
    round(result["response_time"], 2),
    "seconds"
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


CUSTOMER: 7590-VHVEG

QUESTION:

Why is this customer considered at risk,
and what retention policy should support staff consider?


AI ANSWER:
This customer is considered at high risk of churning due to several factors, including their tenure (1 month), monthly charges ($29.85), and payment method (Electronic Check). The churn probability of 0.7568 indicates a significant likelihood of leaving within the next year.

Retention policy considerations for this customer might include:

1. **Reviewing Services**: Ensure there are no issues with the services being offered.
2. **Understanding Billing Concerns**: Address any billing questions or discrepancies promptly.
3. **Contract Conditions**: Verify that the customer understands all terms and conditions of the contract.
4. **Technical Problems**: Investigate and resolve any reported technical issues.
5. **Support**: Offer immediate assistance and ensure the customer feels supported.

The AI system has flagged potential risk factors such as

In [41]:
second_customer_id = customer_df[
    "customerID"
].iloc[10]

result2 = generate_integrated_answer(
    second_customer_id,
    "Explain this customer's churn risk."
)

print(result2["answer"])

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


The customer with the ID 9763-GRSKD has a high churn risk of approximately 35.77%. This is indicated by the churn probability value of 0.3577096824163132, which falls into the "High" category according to the churn risk levels defined in the business rules (where values above 0.3 indicate high risk). The churn prediction is set to 0, indicating that the AI system does not predict that this customer will churn based on the given data.

The churn risk is influenced by several factors, including tenure (length of service), payment method, internet service type, and other demographic characteristics. In this case, the customer has been with the company for 13 months, uses DSL as their internet service, pays by mail check, and has a monthly charge of $


In [40]:
tools = {
    "get_customer_profile": get_customer_profile,
    "predict_churn": predict_customer,
    "get_shap_explanation": get_shap_explanation,
    "search_knowledge_base": search_knowledge_base
}

print("Available tools:")
for tool_name in tools:
    print("-", tool_name)

Available tools:
- get_customer_profile
- predict_churn
- get_shap_explanation
- search_knowledge_base


In [42]:
print("PROFILE")
print(get_customer_profile(test_customer_id))

print("\nCHURN")
print(predict_customer(test_customer_id))

print("\nSHAP")
print(get_shap_explanation(test_customer_id))

print("\nKNOWLEDGE")
print(
    search_knowledge_base(
        "cancellation policy"
    )
)

PROFILE
{'customerID': '7590-VHVEG', 'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'No', 'tenure': 1, 'PhoneService': 'No', 'InternetService': 'DSL', 'Contract': 'Month-to-month', 'PaymentMethod': 'Electronic check', 'MonthlyCharges': 29.85, 'TotalCharges': 29.85, 'PaperlessBilling': 'Yes'}

CHURN
{'customer_id': '7590-VHVEG', 'churn_probability': 0.7567963733310893, 'churn_prediction': 1, 'risk_level': 'High'}

SHAP
                         Feature  SHAP_Value  Absolute_SHAP
5               num__NewCustomer    0.436900       0.436900
1                    num__tenure    0.400227       0.400227
2            num__MonthlyCharges   -0.312355       0.312355
21      cat__InternetService_DSL   -0.153317       0.153317
42  cat__Contract_Month-to-month    0.142881       0.142881

KNOWLEDGE
[{'source': 'cancellation_policy.txt', 'text': "Cancellation Policy Customers may request cancellation of their service. Customers should review their current contract terms before c

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [43]:
integration_config = {
    "ml_model": "../models/improved_logistic_regression.pkl",
    "ml_threshold": 0.55,
    "shap_model": "Logistic Regression",
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_database": "FAISS",
    "llm": MODEL_NAME,
    "tools": [
        "get_customer_profile",
        "predict_churn",
        "get_shap_explanation",
        "search_knowledge_base"
    ]
}

os.makedirs(
    "../models",
    exist_ok=True
)

with open(
    "../models/integration_config.pkl",
    "wb"
) as f:
    pickle.dump(
        integration_config,
        f
    )

print("Integration configuration saved.")

Integration configuration saved.


In [45]:
def customer_intelligence_pipeline(
    customer_id,
    question
):
    
    profile = get_customer_profile(
        customer_id
    )
    
    if profile is None:
        return {
            "error": "Customer not found."
        }
    
    prediction = predict_customer(
        customer_id
    )
    
    shap_info = get_shap_reasons(
        customer_id
    )
    
    knowledge = search_knowledge_base(
        question
    )
    
    llm_result = generate_integrated_answer(
        customer_id,
        question
    )
    
    return {
        "customer_profile": profile,
        "churn_prediction": prediction,
        "shap_reasons": shap_info,
        "knowledge_sources": [
            x["source"]
            for x in knowledge
        ],
        "final_answer": llm_result["answer"],
        "response_time": llm_result["response_time"]
    }